# 🌌 MURE AGI: The Great Distillation (5M Rules)
### Pipeline: Extraction -> Distillation -> Deployment
**Created by Myo Min Aung**

This system handles the autonomous generation of 5 million causal rules and distills the logic from Qwen-2-2B into our specialized Sentence-based LLM.

## 1. Environment & Keep-Alive
from google.colab import drive
import os

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/svo cc brain'
os.makedirs(BASE_DIR, exist_ok=True)

print(f"✅ Working Directory set to: {BASE_DIR}")


In [ ]:
!pip install -q jsonlines
import os
import json
import random
import jsonlines
from tqdm import tqdm

if 'BASE_DIR' not in globals():
    BASE_DIR = '/content/drive/MyDrive/svo cc brain'
    print("⚠️ BASE_DIR was not defined, defaulting.")

RULES_FILE = os.path.join(BASE_DIR, 'rules_synthetic_5M.jsonl')
TARGET_COUNT = 5_000_000

def generate_math_rule():
    a, b = random.randint(-1000, 1000), random.randint(-1000, 1000)
    op = random.choice([('+', lambda x,y: x+y), ('-', lambda x,y: x-y), ('*', lambda x,y: x*y)])
    return {'cause': f"The mathematical expression is {a} {op[0]} {b}.", 'effect': f"The calculated result evaluates to {op[1](a,b)}."}

def generate_logic_rule():
    s = random.choice(['Alice', 'Bob', 'Charlie', 'Delta System'])
    st = random.choice(['active', 'offline', 'verified', 'compromised'])
    a = random.choice(['proceed', 'halt', 'escalate', 'lockdown'])
    return {'cause': f"User status check: {s} is currently {st}.", 'effect': f"The policy engine enforces the action: {a}."}

def generate_code_rule():
    v = random.choice(['x', 'y', 'count', 'total'])
    val = random.randint(1, 100)
    return {'cause': f"A loop increments {v} from 0 to {val} by 1.", 'effect': f"The final value of {v} after the loop terminates is {val}."}

generators = [generate_math_rule, generate_logic_rule, generate_code_rule]

rules_count = 0
if os.path.exists(RULES_FILE):
    print("🔍 Validating existing Rule Dataset...")
    with open(RULES_FILE, 'r') as f:
        for _ in f: rules_count += 1

print(f"📊 Progress: {rules_count:,} / {TARGET_COUNT:,} rules.")

if rules_count < TARGET_COUNT:
    print(f"⚡ Generating {TARGET_COUNT - rules_count:,} more rules...")
    with jsonlines.open(RULES_FILE, mode='a') as writer:
        for _ in tqdm(range(TARGET_COUNT - rules_count)):
            writer.write(random.choice(generators)())
    print("✅ 5,000,000 Rules Ready!")
else:
    print("✅ Rule Generation already complete!")


## 2. Dependencies
Installing specialized libraries for high-speed NLP and training.

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes peft datasets sentencepiece jsonlines


## 3. Persistent 5M Rule Extractions
This cell checks if `rules_5m.json` exists. If not, it starts/resumes extraction from Wikipedia until 5,000,000 rules are collected.

### ⚠️ Note on Rule Persistence
Rule generation and persistence logic is now handled in **Step 1** to ensure memory efficiency and prevent `NameError` during execution.
If you need to re-verify the rules, please check the output of Step 1.


## 4. Distillation Engine Setup
Loading the Teacher (Qwen/Gemma) and Student models.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("🧠 Activating Qwen 1.5B (Open) as Teacher...")
teacher_id = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(teacher_id)
tokenizer.pad_token = tokenizer.eos_token

teacher = AutoModelForCausalLM.from_pretrained(
    teacher_id,
    device_map='auto',
    torch_dtype=torch.float16
)

print("👶 Initializing MURE 3B Student Engine...")
# For demonstration, we simply load another instance or a smaller model as student.
# Here we use Qwen2.5-0.5B-Instruct as student for speed.
student_id = 'Qwen/Qwen2.5-0.5B-Instruct'
student = AutoModelForCausalLM.from_pretrained(
    student_id,
    device_map='auto',
    torch_dtype=torch.float16
)
print("✅ Mind Transfer Pipeline Linked!")


## 5. Knowledge Distillation with Auto-Resume
The core trainer that mimics Qwen's logic using the 5M rules.

In [ ]:
import os
import torch
import json
import jsonlines
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.optim import AdamW

if 'BASE_DIR' not in globals():
    BASE_DIR = '/content/drive/MyDrive/svo cc brain'

class RuleDataset(Dataset):
    def __init__(self, path, tk):
        print(f"📖 Mapping Rule Dataset from: {path}")
        try:
            from datasets import load_dataset
            # Efficient lazy-loading using HuggingFace datasets library
            self.data = load_dataset('json', data_files=path, split='train')
        except Exception as e:
            print(f"❌ Failed to load dataset: {e}")
            raise e
        self.tk = tk

    def __len__(self): return len(self.data)

    def __getitem__(self, i):
        rule = self.data[i]
        # Construct the distillation prompt
        text = f"Cause: {rule['cause']} -> Effect: {rule['effect']}"
        enc = self.tk(text, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
        return enc['input_ids'].squeeze(), enc['attention_mask'].squeeze()

class DistillationTrainer:
    def __init__(self, t, s, base_path):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.teacher = t
        self.student = s
        self.checkpoint_dir = os.path.join(base_path, "checkpoints")
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        
        self.opt = AdamW(self.student.parameters(), lr=4e-5)
        self.step = 0
        self._load_resume()

    def _load_resume(self):
        path = os.path.join(self.checkpoint_dir, "latest_distill.pt")
        if os.path.exists(path):
            print(f"🔍 Found checkpoint: {path}. Resuming...")
            ckpt = torch.load(path, map_location='cpu')
            self.student.load_state_dict(ckpt['model'])
            self.opt.load_state_dict(ckpt['opt'])
            self.step = ckpt.get('step', 0)
            print(f"✅ Resumed distillation from Step {self.step}")
        else:
            print("🆕 No existing checkpoint found. Starting fresh.")
        
        self.student.to(self.device)
        # Re-initialize scheduler relative to current step
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=1000000, eta_min=1e-6)
        if os.path.exists(path) and 'scheduler' in ckpt:
            try:
                self.scheduler.load_state_dict(ckpt['scheduler'])
            except: pass

    def save(self):
        path = os.path.join(self.checkpoint_dir, "latest_distill.pt")
        ckpt = {
            'model': self.student.state_dict(), 
            'opt': self.opt.state_dict(), 
            'scheduler': self.scheduler.state_dict(),
            'step': self.step
        }
        # Save Atomic to prevent file corruption during Colab session drop
        torch.save(ckpt, path + ".tmp")
        os.replace(path + ".tmp", path)
        
        if self.step % 5000 == 0:
            torch.save(self.student.state_dict(), os.path.join(self.checkpoint_dir, f"mure_3b_s{self.step}.pt"))
        
        print(f"💾 Checkpoint recorded at Step {self.step}")

    def train(self, loader):
        self.teacher.eval()
        self.student.train()
        GRAD_ACCUM = 4
        current_batch_in_epoch = self.step % len(loader)
        
        print(f"🚀 Training stats: Total Batches={len(loader)}, Current Step={self.step}")
        pbar = tqdm(total=len(loader), initial=current_batch_in_epoch)
        
        for i, (input_ids, mask) in enumerate(loader):
            # Sequential Resume Skip logic
            if self.step > 0 and i < current_batch_in_epoch:
                if i % 100 == 0: pbar.update(100)
                continue
            
            ids, mask = input_ids.to(self.device), mask.to(self.device)
            with torch.no_grad(): 
                t_logits = self.teacher(ids, attention_mask=mask).logits
            
            s_logits = self.student(ids, attention_mask=mask).logits
            
            # Standard KL-Divergence Loss for Distillation
            batch, seq, vocab = s_logits.shape
            loss = F.kl_div(F.log_softmax(s_logits.reshape(-1, vocab) / 2.0, dim=-1),
                            F.softmax(t_logits.reshape(-1, vocab) / 2.0, dim=-1),
                            reduction='batchmean') * 4.0
            
            (loss / GRAD_ACCUM).backward()
            
            if (i + 1) % GRAD_ACCUM == 0:
                self.opt.step()
                self.scheduler.step()
                self.opt.zero_grad()
            
                self.step += 1
                # FREQUENT CHECKPOINT: Every 200 optimization steps
                if self.step % 200 == 0: 
                    self.save()
            
            if i % 10 == 0:
                loss_val = loss.item() * GRAD_ACCUM # Real Loss
                pbar.set_description(f"Loss: {loss_val:.4f}")
            
            pbar.update(1)
        
        self.save() # Save at end of epoch

print("🚀 Initializing Knowledge Distillation Pipeline...")
dataset = RuleDataset(os.path.join(BASE_DIR, "rules_synthetic_5M.jsonl"), tokenizer)
loader = DataLoader(dataset, batch_size=8, shuffle=False) # Shuffle False is CRITICAL for resume accuracy
trainer = DistillationTrainer(teacher, student, BASE_DIR)

print(f"🔥 Distillation Active. Current Global Step: {trainer.step}")
trainer.train(loader)


## 6. Final Deployment Export
Exports the weights and tokenizers for MURE production server.

In [ ]:
torch.save(student.state_dict(), os.path.join(BASE_DIR, "mure_final_3b_weights.pt"))
tokenizer.save_pretrained(os.path.join(BASE_DIR, "tokenizer"))
print("🏆 Distillation Complete. MURE Brain is ready for deployment.")